# Function 3: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [ ]:
import numpy as np

input_data = np.load('initial_data/function_3/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.754364, 0.233177, 0.303208],
    [0.312229, 0.060777, 0.000904]
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


In [ ]:
output_data = np.load('initial_data/function_3/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    -0.09200841551496666,
    -0.18083748652026374,
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5]])
actual_output = -0.015979341188442648

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 3
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


In [ ]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


## Classification framing: good vs bad outputs

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Define "good" as the best quartile of observed outputs.
# For a very small dataset this gives enough positive labels to fit a classifier.
good_threshold = np.quantile(output_data, 0.25)
good_label = (output_data <= good_threshold).astype(int)

class_df = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
class_df["y"] = output_data
class_df["good_label"] = good_label
class_df["distance_to_threshold"] = np.abs(output_data - good_threshold)
class_df["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
class_df = class_df.sort_values("distance_to_threshold")

print(f"Good/bad threshold: y <= {good_threshold:.6e}")
display(class_df)

print("Support-vector-like observed points:")
display(class_df.head(min(5, len(class_df))))


In [ ]:
X = input_data.copy()
y_cls = good_label
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if len(np.unique(y_cls)) < 2:
    print("Only one class is present, so logistic/SVM classifiers cannot be fitted yet.")
else:
    log_reg = LogisticRegression(class_weight="balanced", random_state=0)
    svm_linear = SVC(kernel="linear", class_weight="balanced", probability=True, random_state=0)
    svm_rbf = SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced", probability=True, random_state=0)

    models = {"logistic": log_reg, "linear_svm": svm_linear, "rbf_svm": svm_rbf}
    for name, model in models.items():
        model.fit(X_scaled, y_cls)
        pred = model.predict(X_scaled)
        print("\n", name)
        print("Confusion matrix:\n", confusion_matrix(y_cls, pred))
        print(classification_report(y_cls, pred, zero_division=0))


In [ ]:
# Boundary / uncertainty search: where classifier is closest to p(good)=0.5.
if len(np.unique(y_cls)) >= 2:
    rng = np.random.default_rng(1)
    candidates = rng.random((30000 if d <= 4 else 60000, d))
    cand_scaled = scaler.transform(candidates)

    rows = []
    for name, model in models.items():
        prob_good = model.predict_proba(cand_scaled)[:, 1]
        uncertainty = np.abs(prob_good - 0.5)
        idx = np.argsort(uncertainty)[:10]
        tmp = pd.DataFrame(candidates[idx], columns=[f"x{i+1}" for i in range(d)])
        tmp["model"] = name
        tmp["p_good"] = prob_good[idx]
        tmp["boundary_score_abs_p_minus_0.5"] = uncertainty[idx]
        rows.append(tmp)

    boundary_points = pd.concat(rows, ignore_index=True)
    display(boundary_points.sort_values("boundary_score_abs_p_minus_0.5").head(20))

    candidate = boundary_points.sort_values("boundary_score_abs_p_minus_0.5").iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(float)
    print("Most boundary-like candidate:", np.round(candidate, 6))
    print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(candidate)))


In [ ]:
# Optional 2D plot
if d == 2 and len(np.unique(y_cls)) >= 2:
    grid_res = 200
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)
    for name, model in models.items():
        prob = model.predict_proba(grid_scaled)[:, 1].reshape(grid_res, grid_res)
        plt.figure(figsize=(7, 6))
        cf = plt.contourf(xx, yy, prob, levels=40)
        plt.colorbar(cf, label="Predicted P(good)")
        plt.contour(xx, yy, prob, levels=[0.5], linewidths=2)
        plt.scatter(input_data[:, 0], input_data[:, 1], c=good_label, edgecolors="black", s=80)
        plt.xlabel("x1"); plt.ylabel("x2"); plt.title(f"Good/bad boundary: {name}")
        plt.show()
